# Tutorial 02 — Model Training

Trains the pathway-masked denoising autoencoder.  
Encodes all cells to the pathway-activation latent `h`, aggregates per  patient per cell-type activation.  
Runs the patient-level classifier under leave-one-out cross-validation.  

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys, tempfile
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score)

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'scripts'))

from utils.config import DEVICE, RANDOM_STATE
from models.iraegis.model_utils import PathwayAE
from models.iraegis.train_utils import (train_ae, precompute_embeddings,
                                      AE_LATENT_DIM, AE_DROPOUT)
from models.iraegis.data_utils import load_cohort_data
from run_iraegis_inference import (aggregate_top25_mean,
                                 gated_ct_stacking_predict)

print(f'PyTorch device: {DEVICE}')

PyTorch device: mps


In [2]:
COHORT = 'GSE189125_pre_ici'
PRIOR  = str(REPO / 'datasets' / 'resources' / 'pathway_prior.npz')
AE_EPOCHS = 80

In [3]:
# Load data
(X, obs, gene_names, ct_groups, ct_ids,
 pat_ids, pat_labels, prior_data) = load_cohort_data(
    COHORT, prior_path=PRIOR, prior_genes_only=True,
    split_ct_groups=['T_cells', 'Monocytes', 'Dendritic'])

n_genes    = X.shape[1]
n_pathways = prior_data['mask'].shape[1]
n_ct       = len(ct_groups)
mask_t     = torch.tensor(prior_data['mask'], dtype=torch.float32)
unique_pats = sorted(pat_labels.keys())
pat_arr     = np.array([pat_labels[p] for p in unique_pats])
pat_id_arr  = np.asarray(pat_ids)

print(f'{X.shape[0]:,} cells, {len(unique_pats)} patients, {n_ct} CTs')

Loading datasets/processed_h5ad/GSE189125_pre_ici.h5ad ...
  Cohort GSE189125_pre_ici: 29,626 cells
[relabel_by_grade] GSE189125_pre_ici: grade>=3 → 18,881 Yes cells, 10,745 No cells (16 patients)
[cell QC] Dropping 1,280/29,626 cells with <200 genes
  Inferring cell-type groups ...
=== Inferred cell type groups ===
  CD8+ NKT-like cells: 15 patients, 1,983 cells
    CD8+ NKT-like cells                                  1983 cells, 15 patients
  Naive CD4+ T cells: 15 patients, 6,972 cells
    Naive CD4+ T cells                                   6972 cells, 15 patients
  Naive CD8+ T cells: 9 patients, 1,068 cells
    Naive CD8+ T cells                                   1068 cells, 9 patients
  B_cells: 14 patients, 4,674 cells
    Naive B cells                                        4674 cells, 14 patients
  NK_cells: 15 patients, 3,620 cells
    Natural killer  cells                                3620 cells, 15 patients
  Classical Monocytes: 15 patients, 4,773 cells
    Classical Mo

In [4]:
# Train the Pathway-Masked Autoencoder
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
ae = PathwayAE(n_genes, n_pathways, mask_t,
               AE_LATENT_DIM, AE_DROPOUT,
               norm='ctbn', act='gelu', n_ct=n_ct).to(DEVICE)

history = train_ae(ae, X, n_epochs=AE_EPOCHS, ct_ids=ct_ids)
print(f'AE training done; final val loss = {history[-1]["val_recon"]:.4f}')

  AE  epoch  10  train=0.7949  val=0.2913  best=0.2913
  AE  epoch  20  train=0.5314  val=0.2773  best=0.2773
  AE  epoch  30  train=0.4257  val=0.2739  best=0.2739
  AE  epoch  40  train=0.3809  val=0.2723  best=0.2722
  AE  epoch  50  train=0.3599  val=0.2713  best=0.2713
  AE  epoch  60  train=0.3504  val=0.2708  best=0.2708
  AE  epoch  70  train=0.3466  val=0.2706  best=0.2706
  AE  epoch  80  train=0.3462  val=0.2706  best=0.2706
AE training done; final val loss = 0.2706


In [5]:
# Compute embeddings from frozen AE
tmpdir = Path(tempfile.mkdtemp(prefix='irAEGIS_h_'))
h, z = precompute_embeddings(ae, X, obs, tmpdir, ct_ids=ct_ids)
print(f'h shape = {h.shape}')

  Saved h (28346, 50), z (28346, 32) to irAEGIS_h_8qb1dxp6/
h shape = (28346, 50)


In [6]:
## 6. Aggregate representation for downstream classification
pat_h = aggregate_top25_mean(h, pat_id_arr, ct_ids, unique_pats, n_ct)
print(f'pat_h shape = {pat_h.shape}')

pat_h shape = (16, 10, 50)


In [7]:
# Patient-level prediction
n_pat = len(unique_pats)
oof_probs = np.zeros(n_pat)
for P_idx in range(n_pat):
    oof_probs[P_idx] = gated_ct_stacking_predict(pat_h, pat_arr, P_idx)

In [8]:
# Performance metrics
auc   = roc_auc_score(pat_arr, oof_probs)
auprc = average_precision_score(pat_arr, oof_probs)
print(f'AUC    = {auc:.4f}')
print(f'AUPRC  = {auprc:.4f}')

AUC    = 0.9524
AUPRC  = 0.9627


In [9]:
out = REPO / 'results' / 'iraegis_oof' / COHORT
out.mkdir(parents=True, exist_ok=True)
np.save(out / 'iraegis_oof_probs.npy',  oof_probs.astype(np.float32))
np.save(out / 'iraegis_oof_labels.npy', pat_arr.astype(np.int8))